# 01 - Load the dataset

**What this does:** loads `data/processed/posts.parquet` — the ONE merged
dataset (7.95M Reddit posts across 15 subreddits 2008-2025, plus ~2.9M X
(Twitter) finance posts covering 2015-2020 and 2022-2024 — note the hole
across 2021 — once `add_x_data.py` has been run; every row carries a
`source` column, 'reddit' or 'x') — and lets you slice it by subreddit and
date without touching the raw files.

**This notebook is the head of the chain.** It saves the filtered slice to
`data/processed/posts_slice.parquet`, and that file is what notebooks
02 (ticker mentions) and 04 (theme mentions) read — so the TIME WINDOW and
SUBREDDITS you set below apply to the whole chain (01 → 02 → 03 and
01 → 04 → 05) without re-typing them anywhere. It also builds the
word-ticker screening table (last section) that notebook 02's extractor
uses to ignore words like LOAN / EDGE / RENT unless written as $cashtags.

**This notebook no longer reads `.zst` files.** The raw-to-parquet conversion
already happened (it's slow — ~3 GB of compressed JSON) and lives in
`data_ingestion/scripts/prep_posts.py` (Reddit) and
`data_ingestion/scripts/add_x_data.py` (X). You only re-run those if the raw
dumps change or you want different cleaning rules. Day-to-day analysis
starts HERE, from the parquet, which loads in seconds.

Because the parquet is *columnar* and stored in subreddit blocks (X rows are
their own `x_twitter` block), we can read just the columns and subreddits we
need instead of the whole 1.15 GB file.

In [31]:
import os, sys
# Find the project root so we can import the src/ package and locate data/.
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
print('project root:', ROOT)

project root: c:\Users\alexd\Desktop\GIC\RetailFlow1


In [32]:
# ============ TIME WINDOW - edit freely ============
# THE single place the analysis window is set: notebooks 02/03/04/05 all
# inherit it through the slice file saved at the bottom of this notebook.
# Keep it narrow to keep runs fast. Rough guide with ALL 15 subreddits:
#   1 year  ~ 700k-2.8M posts -> notebook 02 scan: a few minutes
#   all dates = 7.9M posts    -> 02: ~30-60 min, 04 (themes): ~10-15 min
# NOTE: the X (Twitter) data covers 2015-2020 and 2022-2024, with a HOLE
# across all of 2021 - a window inside the hole (like the default below)
# can't feed the Reddit-vs-X comparison from the slice; notebook 02 then
# falls back to building that comparison from the full dataset itself.
# Set both to None for the full 2008-2025 history.
START_DATE = '2021-01-01'   # inclusive,  'YYYY-MM-DD' or None
END_DATE   = '2022-01-01'   # EXCLUSIVE,  'YYYY-MM-DD' or None
# ====================================================

In [33]:
# ============ PARAMETERS - edit these ============
POSTS_PATH = os.path.join(ROOT, 'data', 'processed', 'posts.parquet')
SUBREDDITS = []      # e.g. ['wallstreetbets', 'stocks'];  [] = ALL (incl. 'x_twitter'
                     # once X data is merged - list it like a subreddit to isolate it)
COLUMNS    = None    # keep None when saving the slice - 02/04 need all columns
SLICE_OUT  = os.path.join(ROOT, 'data', 'processed', 'posts_slice.parquet')
                     # notebooks 02 and 04 READ this file - it is how they
                     # inherit the window above. Set to None to skip saving.
# ==================================================

In [34]:
# Quick metadata peek - reads only the file FOOTER, so it is instant
# even though the file is 1.15 GB.
import pyarrow.parquet as pq

pf = pq.ParquetFile(POSTS_PATH)
print('rows       :', f'{pf.metadata.num_rows:,}')
print('columns    :', pf.schema_arrow.names)
print('row groups :', pf.metadata.num_row_groups)

rows       : 10,821,131
columns    : ['id', 'date', 'author', 'score', 'subreddit', 'title', 'selftext', 'num_comments', 'source']
row groups : 46


In [35]:
# Load the data - with filters pushed down into the parquet reader.
# pyarrow uses each row group's min/max stats to SKIP blocks that can't
# match, so asking for one subreddit reads ~1/30th of the file.

### IMPT: This takes a while but works

import pandas as pd

filters = []
if SUBREDDITS:
    filters.append(('subreddit', 'in', [s.lower() for s in SUBREDDITS]))
if START_DATE:
    filters.append(('date', '>=', START_DATE))
if END_DATE:
    filters.append(('date', '<', END_DATE))

posts = pq.read_table(
    POSTS_PATH,
    columns=COLUMNS,
    filters=filters if filters else None,
).to_pandas()

print('loaded', f'{len(posts):,}', 'posts')
print()
print(posts['subreddit'].value_counts())

# Where did the rows come from? (source = 'reddit' or 'x')
print()
if 'source' in posts.columns:
    print('--- sources in this window ---')
    print(posts.groupby('source')['date'].agg(['count', 'min', 'max']))
else:
    print("(no 'source' column - X data not merged yet; run",
          'data_ingestion/scripts/fetch_x_data.py then add_x_data.py)')

loaded 2,833,008 posts

subreddit
wallstreetbets           1407544
cryptocurrency            680919
bitcoin                   166716
personalfinance           154782
stocks                    103950
pennystocks                81807
investing                  64401
stockmarket                56751
options                    37489
daytrading                 29716
thetagang                  16647
financialindependence      13079
dividends                  10788
valueinvesting              6733
securityanalysis            1615
x_twitter                     71
Name: count, dtype: int64

--- sources in this window ---
          count         min         max
source                                 
reddit  2832937  2021-01-01  2021-12-31
x            71  2021-12-27  2021-12-27


In [36]:
# Preview.
posts.head()

,id,date,author,score,subreddit,title,selftext,num_comments,source
0,ko10pg,2021-01-01,[deleted],0,bitcoin,First Time Saved/ Made Money,[deleted],6,reddit
1,ko12mk,2021-01-01,randum-guy,0,bitcoin,Btc dip to 20k?,Is it possible for Bitcoin to dip to 20k? My b...,19,reddit
2,ko15jt,2021-01-01,Mari0805,119,bitcoin,BTC just had the monthly and yearly close! 202...,Let's see what 2021 brings us. I predict 2021 ...,29,reddit
3,ko171i,2021-01-01,[deleted],0,bitcoin,I believe in Bitcoin.,[deleted],5,reddit
4,ko1877,2021-01-01,[deleted],1,bitcoin,Please help me find a solution with my BTC wal...,[deleted],28,reddit


In [37]:
# Save the slice. Notebooks 02 (tickers) and 04 (themes) read THIS file,
# so they automatically inherit the TIME WINDOW + SUBREDDITS set above -
# change the window here, re-run this notebook, and the chain follows.
if SLICE_OUT:
    posts.to_parquet(SLICE_OUT, index=False)
    print('saved', f'{len(posts):,}', 'posts ->', SLICE_OUT)
    print('window in file:', posts['date'].min(), '->', posts['date'].max())
else:
    print('SLICE_OUT is None - nothing saved. Notebooks 02/04 NEED this file;')
    print('set SLICE_OUT in the PARAMETERS cell and re-run.')

saved 2,833,008 posts -> c:\Users\alexd\Desktop\GIC\RetailFlow1\data\processed\posts_slice.parquet
window in file: 2021-01-01 -> 2021-12-31


## Screen word-tickers - case ratio + wordfreq

Some real tickers are also everyday English words (EDGE, LOAN, RENT, TECH,
EARN...). Bare-caps matching counts those words as phantom mentions, and a
hand-maintained stop list never ends. So we let the data decide, using two
signals:

1. **Case ratio (our own corpus).** English words appear mostly lowercase in
   real posts ("the edge of"); tickers appear mostly ALL-CAPS ("NVDA calls").
   For each candidate we count both forms and compute
   `caps_share = caps / (caps + lower)`.
   Measured on this dataset: EDGE ~ 0.02, LOAN ~ 0.002, NVDA ~ 0.93.
2. **wordfreq (fallback).** If a candidate has fewer than 30 sightings here,
   the ratio is not trustworthy, so we ask how common its lowercase form is
   in general English (zipf scale: "edge" 4.7 = common word, "nvda" 2.1 = not).

**Corpus evidence wins over wordfreq**: SNAP looks like a word to wordfreq
(zipf 4.2) but the corpus measures caps_share ~ 0.56 - people write SNAP
like a ticker - so it stays. That is why signal 1 is checked first.

Word-tickers are **demoted, not deleted**: they still count when written as
a cashtag ($LOAN), so a real attention spike is never lost - only the prose
noise is. `src/extract_tickers.py` loads the CSV automatically at import;
notebook 02 needs no changes.

Caveats (details in README "Screening word-tickers"): caps-typed jargon
(HODL) passes the case test - the manual stop list still covers those; and
brand-name tickers people type lowercase (SOFI, HOOD, COIN) get demoted -
that trades recall for precision, the right trade before Stage 2.

In [38]:
# ============ SCREENING PARAMETERS - edit freely ============
SCREEN_SAMPLE_SIZE = 300_000   # posts used for the case-ratio count (~40 s)
CLASSIFICATION_OUT = os.path.join(ROOT, 'data', 'reference', 'ticker_classification.csv')
# The thresholds (MIN_SIGHTINGS=30, CAPS_SHARE_THRESHOLD=0.5,
# ZIPF_WORD_THRESHOLD=3.5) live in src/screen_tickers.py - tune them there
# if the tables below look wrong, then re-run this section.
# ============================================================

In [39]:
# Classify every 4-5 letter ticker in the universe (only those can collide
# with the bare-caps regex WORD_BARE). Uses the posts loaded above, so the
# ratio is measured on the SAME time window / subreddits you are analysing
# (X rows, if present in the window, are part of the sample too).
from pathlib import Path
from src.screen_tickers import screen_tickers
from src.ticker_universe import load_us_ticker_universe

universe = load_us_ticker_universe(
    Path(ROOT) / 'data' / 'reference',  # cached Nasdaq symbol files
    max_cache_age_days=365,             # use the cache; avoids re-downloading
)
candidates = {t for t in universe if 4 <= len(t) <= 5}
print('candidate tickers to screen:', len(candidates))

# A sample is plenty - 300k posts already sights common words thousands
# of times. random_state=0 makes the sample (and the CSV) reproducible.
sample = posts.sample(min(SCREEN_SAMPLE_SIZE, len(posts)), random_state=0)
texts = (sample['title'].fillna('') + ' ' + sample['selftext'].fillna('')).tolist()

classification = screen_tickers(texts, candidates)   # ~40 s for 300k posts
classification.to_csv(CLASSIFICATION_OUT, index=False)

n_demoted = (classification['classification'] == 'cashtag_only').sum()
print('saved ->', CLASSIFICATION_OUT)
print(n_demoted, 'tickers demoted to cashtag-only |',
      len(classification) - n_demoted, 'kept normal')

candidate tickers to screen: 9349
saved -> c:\Users\alexd\Desktop\GIC\RetailFlow1\data\reference\ticker_classification.csv
418 tickers demoted to cashtag-only | 8931 kept normal


In [40]:
# Sanity-check the decisions.
# Table 1: worst word-ticker offenders we just demoted - a huge lower_count
# means the word is everywhere in prose, exactly what was polluting counts.
seen = classification[classification['caps_count'] + classification['lower_count'] >= 30].copy()
demoted = seen[seen['classification'] == 'cashtag_only']
print('--- top demoted word-tickers ---')
print(demoted.sort_values('lower_count', ascending=False).head(15).to_string(index=False))

# Table 2: the borderline zone - the only rows worth eyeballing by hand.
# If one looks wrong, add it to the manual stop lists in extract_tickers.py
# (definitely a word) or leave it - its $cashtag mentions always count.
borderline = seen[(seen['caps_share'] > 0.35) & (seen['caps_share'] < 0.65)].copy()
borderline['total'] = borderline['caps_count'] + borderline['lower_count']
print()
print('--- borderline (0.35 < caps_share < 0.65), most-seen first ---')
print(borderline.sort_values('total', ascending=False).head(15).to_string(index=False))

--- top demoted word-tickers ---
ticker  caps_count  lower_count  caps_share  zipf decided_by classification
  JUST         530        32462       0.016  6.43 case_ratio   cashtag_only
  TIME         615        20032       0.030  6.29 case_ratio   cashtag_only
  KNOW         186        16635       0.011  6.10 case_ratio   cashtag_only
  HERE        2002        16617       0.108  5.97 case_ratio   cashtag_only
  YEAR          76        15776       0.005  5.96 case_ratio   cashtag_only
  GOOD         182        13756       0.013  6.12 case_ratio   cashtag_only
  WANT         181        12581       0.014  6.04 case_ratio   cashtag_only
  NEXT         530        11012       0.046  5.70 case_ratio   cashtag_only
  HELP         447        10564       0.041  5.75 case_ratio   cashtag_only
  WEEK         376         8723       0.041  5.56 case_ratio   cashtag_only
  WELL          70         7792       0.009  6.03 case_ratio   cashtag_only
  HOLD        3945         7541       0.343  5.19 case_